In [1]:
from s2_simu_mcmc_grw_nf import *

In [2]:
# 1 #####################################################################################################
# Set-up
target = 'm_grw_nf'

# initializing models
IPM_true = GP_IPM(popu_data=popu_dataset, GPmodel_true=models_true)
IPM_pret_grw_nf = Perted_IPM(popu_data=popu_dataset, GPmodel_true=models_true, target=target, NB=False)

# Firstly, we build the GPR model.
kernel = gpflow.kernels.RBF(lengthscales=np.array([5,50]))
m_grw_nf_new = gpflow.models.GPR(data=(IPM_pret_grw_nf.X_grw_nf, IPM_pret_grw_nf.Y_grw_nf), 
                                kernel=kernel, mean_function=None)

# Secondly,we add priors to the hyperparameters.
m_grw_nf_new.kernel.variance.prior = tfd.InverseGamma(f64(0.001),f64(0.001))
m_grw_nf_new.likelihood.variance.prior = tfd.InverseGamma(f64(0.001),f64(0.001))
m_grw_nf_new.kernel.lengthscales.prior = tfd.Normal(loc=f64(0.), scale=f64(100.))

#########################################################################################################


In [ ]:
# 2 #####################################################################################################
# We now read samples
hmc_helper = gpflow.optimizers.SamplingHelper(
    m_grw_nf_new.log_posterior_density, m_grw_nf_new.trainable_parameters
)
num_burnin_steps = 30000
num_samples = 20000

# samples = pickle.load(open(file = os.getcwd()+"/mcmc_samples/m_grw_nf" +"/samples.pkl", mode="rb"))
parameter_samples = pickle.load(open(file = os.getcwd()+"/mcmc_samples/m_grw_nf" +"/parameter_samples.pkl", mode="rb"))

    #  assign samples to our models
IPM_pret_grw_nf.mcmc_para_sample = parameter_samples

#########################################################################################################

In [5]:
# 3 #####################################################################################################
# loading populations which were generated by MCMC samples
print('Loading summary_data for all the MCMC samples')
summary_data = pickle.load(open(file = os.getcwd() + "/mcmc_samples/m_grw_nf" + "/summary_data.pkl", mode="rb"))
IPM_pret_grw_nf.nlog_post = np.array(summary_data['nlpo'])
IPM_pret_grw_nf.nlog_likeli = np.array(summary_data['nll'])

#########################################################################################################

# 4 #####################################################################################################
# Now, if we consider the estimates with the likelihood scores around the optimum.
#      Calculating the summary tests around the optimum for full data.
rep = 1000
opt_percentage = 2
print('\nLoading summary_data for all the MCMC samples around the optimum')
summary_opt = pickle.load(open(file = os.getcwd() + "/mcmc_samples/m_grw_nf" + "/summary_opt.pkl", mode="rb"))
summary_around_opt = pickle.load(open(file = os.getcwd() + "/mcmc_samples/m_grw_nf" + "/summary_around_opt.pkl", mode="rb"))


Loading summary_data for all the MCMC samples

Loading summary_data for all the MCMC samples around the optimum


In [8]:
# 5 #####################################################################################################
# find the top ten stats which are most sens for this kind of perturbation
IPM_pret_grw_nf.opt_percentage=opt_percentage
print('\n\n\n' + 'Re-calculating the top summary stats' + '\n\n\n')
num_mcmc = IPM_pret_grw_nf.mcmc_para_sample[0].shape[0]

most_freq_summary_stats = pd.DataFrame(data=0.0, 
                                        index=range(np.sum(IPM_pret_grw_nf.whether_around_opt_comp)), 
                                        columns=IPM_pret_grw_nf.col_names)
auc_data = pd.DataFrame(data=0.0, index=range(np.sum(IPM_pret_grw_nf.whether_around_opt_comp)), columns=IPM_pret_grw_nf.col_names)

for j in range(int(IPM_pret_grw_nf.mcmc_para_sample[0].shape[0]*opt_percentage/100)):
    d = summary_around_opt.loc[(0+j*rep):(rep-1+j*rep)].reset_index(drop=True).copy()
    auc0 = np.zeros(76)
    auc1 = np.zeros(76)
    for i in range(76):        
        fpr0, tpr0, _ = roc_curve(y_true=np.append(np.repeat(1, rep), np.repeat(0, rep)), 
                                y_score=np.append(summary_opt.iloc[:, i], d.iloc[:, i]), pos_label=0)
        auc0[i] = auc(fpr0, tpr0)
        
    auc_data.iloc[j] = auc0  
    most_freq_summary_stats.iloc[j, np.argsort(auc0)[np.sort(auc0) > 0.75]] = most_freq_summary_stats.iloc[j, np.argsort(auc0)[np.sort(auc0) > 0.75]]+ 1






Re-calculating the top summary stats





In [10]:
print(np.sort(most_freq_summary_stats.sum())[-5:])
most_columns = most_freq_summary_stats.columns[np.argsort(most_freq_summary_stats.sum())[-5:]]
print(most_columns)


[266. 299. 301. 301. 309.]
Index(['27g', '27f', '27d', '27c', '27e'], dtype='object')
